## Imports

In [21]:
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import numpy as np
import evaluate

# Load dataset

In [22]:
dataset = load_dataset("stanfordnlp/imdb")

# 2. Take only 10000 samples (e.g. from train split)
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(10000))
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(1000))

## Load tokenizer and model

In [23]:

# 3. Tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


# 4. Load BERT model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15168.58it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

## Freeze

In [24]:
for param in model.bert.parameters():
    param.requires_grad = False

In [26]:
# (optional) check trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable} / {total}")

Trainable: 1538 / 109483778


## Tokenize

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], 
    truncation=True, 
    padding="max_length", 
    max_length=256)

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

## Training args

In [27]:
args = TrainingArguments(
    output_dir="./bert_frozen_imdb",
    learning_rate=5e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
)

## Trainer

In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)


trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
)

## Training 

In [31]:
# 9. Train
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.599777,0.607906,0.671000
2,0.575758,0.554069,0.744000
3,0.565912,0.554231,0.731000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.53it/s]


TrainOutput(global_step=1875, training_loss=0.5899444178263347, metrics={'train_runtime': 80.1676, 'train_samples_per_second': 374.216, 'train_steps_per_second': 23.389, 'total_flos': 3946665830400000.0, 'train_loss': 0.5899444178263347, 'epoch': 3.0})

## Freezing 

In [ ]:
for param in bert.parameters():
    param.requires_grad = False

bert.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

## Convert text → BERT embeddings

In [13]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

tokenized = dataset.map(tokenize, batched=True)
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Map: 100%|██████████| 50000/50000 [00:09<00:00, 5493.53 examples/s]


## Build dataset of embeddings

In [14]:
def get_bert_features(batch):
    with torch.no_grad():
        outputs = bert(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )
        cls_embedding = outputs.last_hidden_state[:, 0, :]
    return cls_embedding

## Train a simple classifier

In [15]:
import torch.nn as nn

classifier = nn.Linear(768, 2)  # IMDb = binary classification

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(tokenized["train"], batch_size=16, shuffle=True)

optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

classifier.train()

for epoch in range(2):
    for batch in train_loader:
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["label"]

        with torch.no_grad():
            outputs = bert(input_ids=input_ids,
                           attention_mask=attention_mask)
            features = outputs.last_hidden_state[:, 0, :]

        logits = classifier(features)

        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("epoch", epoch, "loss", loss.item())

## Evaluate

In [ ]:
X_test = []
y_test = []

for i in range(500):
    text = test_df.iloc[i]["text"]
    label = test_df.iloc[i]["label"]

    X_test.append(get_embedding(text))
    y_test.append(label)

X_test = np.array(X_test)

accuracy = clf.score(X_test, y_test)
print("Accuracy:", accuracy)